1-data importing
2-preprocessing
3-feature engineering
4-train test split +scaling
5-scaling(breaking numbers into smaller numbers. i..e: 455555=0.2)
6-model building
7-model training
8-prediction
9-deployment



#Salary Prediction Regression with Neural Network

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle


In [2]:
data=pd.read_csv('Churn_Modelling.csv')


In [3]:
data.head(1)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.0,1,1,1,101348.88,1


In [4]:
data=data.drop(['CustomerId','Surname','RowNumber'],axis=1)

In [5]:
data.head(1)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.0,1,1,1,101348.88,1


In [6]:
Label_encoder=LabelEncoder()
data['Gender']=Label_encoder.fit_transform(data['Gender'])

In [ ]:
data['Gender']

In [8]:
onehot_encode_geo=OneHotEncoder()
geo_encoder=onehot_encode_geo.fit_transform(data[['Geography']]).toarray()

In [9]:
onehot_encode_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [10]:
geo_encoded_df=pd.DataFrame(geo_encoder,columns=onehot_encode_geo.get_feature_names_out(['Geography']))


In [11]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

In [12]:
with open('Label_encoder.pkl','wb') as file:
    pickle.dump(Label_encoder,file)

with open('onehot_encode_geo.pkl','wb') as file:
    pickle.dump(onehot_encode_geo,file)

In [33]:
data.head(2)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0


In [13]:
data=data.drop(['Exited'],axis=1)

In [ ]:
data.head(1)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.0,1,1,1,101348.88,1.0,0.0,0.0


In [15]:
X=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

In [32]:
X.head(1)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.0,1,1,1,1.0,0.0,0.0


In [16]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)#fittransform is only for training data, not testing one
X_test=scaler.transform(X_test)

In [18]:
scaler_y = StandardScaler()
# We use .values.reshape(-1, 1) because the scaler expects a 2D array
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

In [19]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)
with open('scaler_y.pkl','wb') as file:
    pickle.dump(scaler_y,file)

In [20]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [ ]:
X_train.shape[1]
data.shape[1]

11

In [22]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)  # <--- IMPORTANT: No activation function here for regression
])

c:\Users\shayan\Desktop\datascience_and_ai_class\salary_prediction_regression_dl\venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

In [23]:
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.0001)

In [24]:
loss=tensorflow.keras.losses.MeanSquaredError()
model.compile(optimizer=opt, loss='mse', metrics=['mae'])

In [25]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [26]:
log_dir='logs/fit' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [27]:
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=20,restore_best_weights=True)

In [ ]:
history=model.fit(
    X_train,y_train_scaled, validation_data=(X_test,y_test_scaled),epochs=200,
    callbacks=[early_stopping_callback])

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots
plt.figure(figsize=(14, 5))

# 1. Plot Loss (Mean Squared Error)
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Model Loss (MSE) - Convergence')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 2. Plot MAE (Mean Absolute Error)
plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE', color='green')
plt.plot(history.history['val_mae'], label='Validation MAE', color='red')
plt.title('Model MAE - Error in Dollars')
plt.xlabel('Epochs')
plt.ylabel('MAE ($)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [30]:
model.save('model.h5')

In [31]:
model.save('model.keras')